Master the advanced patterns that unlock the full power of Claude Code -- spawning independent subagents for parallel research, building reusable plugins for your team, and orchestrating automated workflows with lifecycle hooks.
1. The Subagent Pattern
2. Plugin Architecture
4. Hook Lifecycle
6. Skills Packaging and Sharing

In [ ]:
# Agent Tool Call
// Parent's internal tool call:
Agent("Research the authentication flow in this codebase.
Map out every file involved in the auth process:
- Where are credentials validated?
- How are sessions managed?
- What middleware handles token verification?
- Are there any security concerns?

Return a structured summary with file paths and findings.")

// Subagent works independently -- reads 30+ files, greps for patterns,
// traces imports -- all in its own context window.

// Parent receives only this summary:
Authentication Flow Summary:
- Entry point: src/middleware/auth.ts (verifyToken middleware)
- Token validation: src/services/jwt.ts (RS256 with key rotation)
- Session store: src/services/session.ts (Redis-backed, 24h TTL)
- User lookup: src/models/user.ts -> src/db/queries/auth.sql
- 3 security concerns identified:
  1. No rate limiting on /api/login endpoint
  2. Refresh tokens stored in localStorage (XSS risk)
  3. Missing CSRF protection on session cookie

In [ ]:
# settings.json
{
  "hooks": {
    "PreToolUse": [
      {
        "matcher": "Bash",
        "command": "./hooks/check-safety.sh $TOOL_INPUT"
      },
      {
        "matcher": "Write",
        "command": "./hooks/check-write-target.sh $TOOL_INPUT"
      }
    ],
    "PostToolUse": [
      {
        "matcher": "Edit",
        "command": "./hooks/validate-format.sh $TOOL_INPUT"
      }
    ],
    "Stop": [
      {
        "command": "./hooks/check-completion.sh"
      }
    ]
  }
}


In [ ]:
# hooks/check-completion.sh
#!/bin/bash
# Stop hook: check if all phases are complete

STATE_FILE=".jumpstarter-state.json"

# If no state file, allow stop
if [ ! -f "$STATE_FILE" ]; then
  exit 0
fi

# Read state values
CURRENT=$(python3 -c "import json; print(json.load(open('$STATE_FILE'))['current_phase'])")
TOTAL=$(python3 -c "import json; print(json.load(open('$STATE_FILE'))['total_phases'])")
ITERATION=$(python3 -c "import json; print(json.load(open('$STATE_FILE'))['iteration_count'])")
MAX_ITER=$(python3 -c "import json; print(json.load(open('$STATE_FILE'))['max_iterations'])")

# Safety: always stop if max iterations reached
if [ "$ITERATION" -ge "$MAX_ITER" ]; then
  echo "Max iterations reached ($ITERATION/$MAX_ITER). Allowing stop."
  exit 0
fi

# If more phases remain, keep going
if [ "$CURRENT" -lt "$TOTAL" ]; then
  echo "Phase $CURRENT of $TOTAL complete. Continue to next phase." >&2
  exit 1
fi

# All phases complete
echo "All $TOTAL phases complete!"
exit 0

In [ ]:
 # Example: Database Migration Skill
 ---
trigger: "when creating database migrations or modifying schema"
context: "fork"
---

# Database Migration Skill

## Rules
1. Every migration MUST have a corresponding rollback
2. Use timestamped filenames: YYYYMMDD_HHMMSS_description.sql
3. Never use DROP TABLE in production -- use soft deletes
4. Always add NOT NULL constraints with DEFAULT values
5. Create indexes for any column used in WHERE or JOIN

## Migration Template
See operations/migration-template.sql for the standard format.

## Testing Requirements
- Run migration forward AND backward before committing
- Verify with: `psql -f migration.sql && psql -f rollback.sql`
- Check for data loss with: SELECT COUNT(*) before and after